## 🤖 Semantic Model AI Readiness Analyzer v1.0

**For Power BI Copilot and AI-Powered Analytics**

### What This Analyzer Does

- ✅ **Validates your semantic model** for AI-powered analytics (Copilot in Power BI)
- ✅ **Automates 18+ critical checks** — descriptions, naming, DAX quality, performance, AI Data Schema
- ✅ **Actionable scoring** — severity-weighted priorities (🔴 Critical → 🟡 Recommended)
- ✅ **Prep for AI guidance** — Verified Answers, AI Instructions, synonyms, row labels
- ✅ **100% aligned** with official Microsoft guidance for AI-ready semantic models

### Version History

**v1.0**
- Security requirements in scoping assessment
- Performance Analyzer for AI Data Schema measures
- Duplicate column names detection across tables
- Default groupings in AI Instructions
- Scoping & Design assessment for focused solutions
- Row Label validation for dimension tables
- Direct Lake optimization guidance
- Report-scoped measures detection & data type validation
- Testing & Validation framework with severity-weighted scoring

### Coverage Matrix

| # | Check Category | Automation | Importance |
|---|----------------|------------|------------|
| 1 | **Semantic Model Optimization** | | |
| 1.1 | Star schema validation | ✅ Automated | 🔴 Critical |
| 1.2 | Best Practice Analyzer | ✅ Automated | 🔴 Critical |
| 1.3 | Memory Analyzer | 🔵 Manual | 🟠 Important |
| 1.4 | Direct Lake optimization | 🔵 Manual | 🟡 Conditional |
| 1.5 | Data type validation | ✅ Automated | 🟠 Important |
| 1.6 | Unnecessary objects | ✅ Automated | 🟡 Recommended |
| 1.7 | Business-friendly naming | ✅ Automated | 🔴 Critical |
| 1.8 | Descriptions coverage | ✅ Automated | 🔴 Critical |
| 1.9 | Synonyms configuration | ✅ Automated | 🟡 Recommended |
| 1.10 | Row labels | ✅ Automated | 🟠 Important |
| 1.11 | Explicit measures | ✅ Automated | 🔴 Critical |
| 1.12 | Ambiguous date fields | ✅ Automated | 🟠 Important |
| 1.13 | Hidden objects risk | ✅ Automated | 🟠 Important |
| 1.14 | Model complexity / bloat | ✅ Automated | 🟡 Recommended |
| 1.15 | Report-scoped measures | ✅ Automated | 🟠 Important |
| 1.16 | Duplicate measures | ✅ Automated | 🟠 Important |
| 1.17 | Performance Analyzer (AI Schema) | 🔵 Manual | 🟠 Important |
| 1.18 | Duplicate column names | ✅ Automated | 🟠 Important |
| 0 | **Scoping & Design** | | |
| 0.6 | Security requirements | 🔵 Manual | 🔴 Critical |
| 2 | **Prep for AI — AI Data Schema** | | |
| 2.1 | Schema selection validation | 🔵 Manual | 🔴 Critical |
| 2.2 | Measure dependencies | ✅ Automated | 🔴 Critical |
| 2.3 | Helper measures exclusion | ✅ Automated | 🟠 Important |
| 2.4 | Hidden fields in verified answers | ✅ Automated | 🔴 Critical |
| 3 | **Prep for AI — Verified Answers** | 🔵 Manual | 🔴 Critical |
| 4 | **Prep for AI — AI Instructions** | 🔵 Manual | 🔴 Critical |
| 4.7 | Default groupings & analysis preferences | 🔵 Manual | 🟡 Recommended |
| 5 | **Copilot & AI Testing** | 🔵 Manual | 🔴 Critical |
| 6 | **Testing & Validation** | 🔵 Manual | 🔴 Critical |

> **Powered by** [Semantic Link](https://learn.microsoft.com/fabric/data-science/semantic-link-overview), [Semantic Link Labs](https://github.com/microsoft/semantic-link-labs), and [Microsoft Prep for AI](https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai) guidance

### 📋 Prerequisites
- Run inside a **Microsoft Fabric** workspace notebook
- The semantic model must be published to a Fabric workspace
- You need **Build** (or higher) permissions on the semantic model
- `semantic-link-labs` will be installed automatically in the next cell
- **Recommended:** Run Best Practice Analyzer and Memory Analyzer before this notebook

In [ ]:
# Install required packages
%pip install semantic-link-labs --quiet --upgrade

### ⚙️ Configuration — Set Your Parameters
Update the values below, then **Run all cells**.

In [ ]:
import sempy.fabric as fabric
import pandas as pd
import re
import warnings
import json
from datetime import datetime
warnings.filterwarnings('ignore')
from IPython.display import display, HTML, Markdown

# ============================================================
# 🔧  PARAMETERS — update these values
# ============================================================
dataset   = "ContosoSalesData"           # Name or ID of your semantic model
workspace = "YourWorkspaceName"          # Name or ID of your Fabric workspace

# Optional: Set to True if you've already configured Prep for AI
prep_for_ai_configured = False

# Optional: Set to True if this is a Direct Lake model
is_direct_lake = False
# ============================================================

print("=" * 70)
print("  🤖 SEMANTIC MODEL AI READINESS ANALYZER v1.0")
print("=" * 70)
print(f"\n  Model     : {dataset}")
print(f"  Workspace : {workspace}")
print(f"  Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n" + "=" * 70)
print("\nStarting analysis ...\n")

### 📦 Load Semantic Model Metadata

In [ ]:
print("Loading semantic model metadata ...")
print("-" * 70)

try:
    tables_df        = fabric.list_tables(dataset=dataset, workspace=workspace)
    columns_df       = fabric.list_columns(dataset=dataset, workspace=workspace)
    measures_df      = fabric.list_measures(dataset=dataset, workspace=workspace)
    relationships_df = fabric.list_relationships(dataset=dataset, workspace=workspace)
except Exception as e:
    print(f"❌ ERROR loading metadata: {e}")
    print("\nVerify:")
    print("  1. Dataset and workspace names are correct")
    print("  2. You have Build permissions on the semantic model")
    print("  3. The semantic model is published to Fabric")
    raise

# Helper functions
def is_hidden_mask(df, col='Is Hidden'):
    if col not in df.columns:
        return pd.Series([False] * len(df), index=df.index)
    return df[col].apply(lambda v: str(v).lower() == 'true' if not isinstance(v, bool) else v)

def missing_desc_mask(df, col='Description'):
    if col not in df.columns:
        return pd.Series([True] * len(df), index=df.index)
    return df[col].isna() | (df[col].astype(str).str.strip() == '')

# Filter system tables
SYS_PATTERN = r'^(DateTableTemplate_|LocalDateTable_)'
system_mask = tables_df['Name'].str.match(SYS_PATTERN, na=False)
tables_df   = tables_df[~system_mask].copy()

# Separate visible and hidden
hidden_tbl_mask = is_hidden_mask(tables_df)
hidden_col_mask = is_hidden_mask(columns_df)
hidden_msr_mask = is_hidden_mask(measures_df)

visible_tables   = tables_df[~hidden_tbl_mask]
visible_columns  = columns_df[~hidden_col_mask]
visible_measures = measures_df[~hidden_msr_mask]

# Remove row-number columns
if 'Is Row Number' in columns_df.columns:
    rn_mask = columns_df['Is Row Number'].apply(lambda v: str(v).lower() == 'true' if not isinstance(v, bool) else v)
    visible_columns = visible_columns[~rn_mask[visible_columns.index]]

print(f"✅ Metadata loaded for: '{dataset}'")
print()
print(f"  Visible tables   : {len(visible_tables):>4}  (total incl. hidden: {len(tables_df)})")
print(f"  Visible columns  : {len(visible_columns):>4}  (total incl. hidden: {len(columns_df)})")
print(f"  Visible measures : {len(visible_measures):>4}  (total incl. hidden: {len(measures_df)})")
print(f"  Relationships    : {len(relationships_df):>4}")
print()

# Global trackers
check_scores = {}  # {check_key: {'achieved': n, 'max': n, 'weight': n}}
all_issues   = []  # [(severity, check, description)]  # severity: critical | important | recommended

---
# 0️⃣ SCOPING & DESIGN ASSESSMENT

**Before diving into technical validation, ensure your AI-powered solution has a clear, narrow scope.**

> ⚠️ **Key Insight from Microsoft:** "AI solutions with laser-focused missions deliver value. General-purpose 'everything' solutions fall short due to performance issues, confused logic, low trust, and maintenance challenges."

### Why Narrow Scope Matters

**Avoid:** ❌ General-purpose solutions covering:
- Sales & Revenue Analytics
- Inventory & Supply Chain  
- Marketing & Campaign Performance
- Customer Support & Experience
- Finance & Forecasting
- Order Fulfillment & Logistics

**Prefer:** ✅ Domain-specific solutions like:
- **Sales Intelligence** — real-time order data, sales KPIs only
- **RevOps Intelligence** — marketing campaigns + sales correlation only
- **Customer Experience** — support tickets + satisfaction metrics only

### Scoping Checklist (Manual Assessment Required)

Before deploying AI-powered analytics (Copilot), answer these questions:


In [ ]:
print("=" * 70)
print("CHECK 0: SCOPING & DESIGN ASSESSMENT (MANUAL)")
print("=" * 70)
print("\n🎯 CRITICAL: Complete this scoping exercise BEFORE technical configuration.\n")
print("✅ SCOPING CHECKLIST:")
print()
print("   📝 1. WHO ARE THE PRIMARY USERS?")
print("      Define specific user personas (e.g., 'sales analysts', 'regional managers')")
print("      NOT generic 'business users' or 'everyone in the company'")
print()
print("   📝 2. WHAT ARE THE MAIN FUNCTIONS? (List 3-5 specific tasks)")
print("      Examples:")
print("        ✅ 'Monitor real-time order volume by region'")
print("        ✅ 'Compare campaign ROI across marketing channels'")
print("        ✅ 'Identify top complaint categories by geography'")
print("      NOT:")
print("        ❌ 'Answer any business question'")
print("        ❌ 'Provide insights from all data'")
print()
print("   📝 3. TOP 10-15 QUESTIONS USERS WILL ASK")
print("      List actual expected questions:")
print("        • 'What are our top-selling products this week?'")
print("        • 'Which warehouses are running low on stock?'")
print("        • 'What's the ROI of Q1 influencer campaigns in APAC?'")
print()
print("   📝 4. WHAT'S OUT OF SCOPE?")
print("      Clearly define what AI features WON'T handle:")
print("        • HR/employee queries → not included")
print("        • Financial forecasting → separate solution")
print("        • Unstructured documents → not in scope")
print()
print("   📝 5. DATA AVAILABILITY CHECK")
print("      For each key question, verify:")
print("        ☐ Data exists in Fabric (Lakehouse/Warehouse/Semantic Model)")
print("        ☐ Data is current (refresh frequency meets requirements)")
print("        ☐ Data quality is acceptable (no major gaps/errors)")
print("        ☐ Questions are answerable with existing data")
print()
print("   📝 6. SECURITY REQUIREMENTS")
print("      Define who can access AI-powered analytics:")
print("        ☐ Identify user personas who will use Copilot/AI features")
print("        ☐ Document Row Level Security (RLS) requirements")
print("        ☐ List sensitive data fields to exclude from AI Data Schema")
print("        ☐ Confirm compliance with data privacy policies (GDPR, HIPAA, etc.)")
print("        ☐ Define if solution should be internal-only or externally accessible")
print("        ☐ Verify workspace-level permissions are appropriately scoped")
print()
print("⚠️  WARNING SIGNS OF SCOPE PROBLEMS:")
print("   • More than 20 tables selected in AI Data Schema")
print("   • Solution serves multiple unrelated business domains")
print("   • Users from 5+ different departments")
print("   • Mix of real-time and historical analytics")
print("   • Questions require 'why' or open-ended analysis")
print()
print("💡 DESIGN PATTERNS:")
print()
print("   ┌─────────────────────────────────────────────────────────┐")
print("   │ PATTERN 1: Focused Semantic Models (RECOMMENDED)       │")
print("   ├─────────────────────────────────────────────────────────┤")
print("   │ • Sales Analytics Model → Order data + Sales KPIs only │")
print("   │ • Marketing Model → Campaign data + ROI metrics only    │")
print("   │ • Customer Experience Model → Support + CSAT only      │")
print("   │                                                         │")
print("   │ Benefits: High accuracy, clear ownership, maintainable │")
print("   └─────────────────────────────────────────────────────────┘")
print()
print("   ┌─────────────────────────────────────────────────────────┐")
print("   │ PATTERN 2: Composite Star Schema (ADVANCED)            │")
print("   ├─────────────────────────────────────────────────────────┤")
print("   │ Main semantic model with:                              │")
print("   │   → Sales fact                                          │")
print("   │   → Marketing fact                                      │")
print("   │   → Customer Support fact                               │")
print("   │   → Shared dimension tables                             │")
print("   │                                                         │")
print("   │ Use Prep for AI to configure separate AI Data Schemas  │")
print("   │ for different question domains                          │")
print("   └─────────────────────────────────────────────────────────┘")
print()
print()

# Score based on awareness
score = 10  if prep_for_ai_configured else 0
check_scores['0_scoping'] = {'achieved': score, 'max': 10, 'weight': 2}
if not prep_for_ai_configured:
    all_issues.append(("critical", "0 Scoping", "Scoping questionnaire not completed"))
    
print(f"📊 Score: {score}/10 (manual assessment required)")
print()
print("=" * 70)

---
# 1️⃣ SEMANTIC MODEL OPTIMIZATION

This section validates foundational model quality that directly impacts AI accuracy and performance (Copilot).

## 1.1 ⭐ Star Schema Validation

**Why it matters:** AI-powered DAX generation (Copilot) relies on clear fact-dimension relationships. Flat/denormalized models produce unreliable results.

**What's checked:**
- No relationships (flat model) — **CRITICAL**
- Many-to-many relationships — **should be rare**
- Bidirectional cross-filtering — **use sparingly**
- Isolated visible tables — **connect or hide**

In [ ]:
print("=" * 70)
print("CHECK 1.1: STAR SCHEMA VALIDATION")
print("=" * 70)

issues = []
score = 20  # Max points

if relationships_df.empty:
    print("❌ CRITICAL: No relationships found — flat/denormalized model detected.")
    print("\n   This is a BLOCKING issue for AI accuracy.")
    print("   Refactor into a star schema with clear fact and dimension tables.")
    print("\n   📚 Learn: https://learn.microsoft.com/power-bi/guidance/star-schema")
    issues.append(("critical", "1.1 Star Schema", "No relationships found — flat model"))
    score = 0
else:
    # Many-to-many relationships
    m2m_col = next((c for c in ['Multiplicity', 'Cardinality'] if c in relationships_df.columns), None)
    if m2m_col:
        m2m = relationships_df[
            relationships_df[m2m_col].astype(str).str.contains('ManyToMany|Many.*Many', case=False, na=False)
        ]
        if not m2m.empty:
            print(f"⚠️  {len(m2m)} many-to-many relationship(s) found:")
            for _, r in m2m.iterrows():
                ft = r.get('From Table', 'Unknown')
                tt = r.get('To Table', 'Unknown')
                print(f"     • {ft}  ↔  {tt}")
            print("\n   M:M relationships reduce DAX accuracy and performance.")
            print("   ACTION: Introduce bridge tables to resolve these.")
            issues.append(("important", "1.1 Star Schema", f"{len(m2m)} many-to-many relationship(s)"))
            score -= min(6, len(m2m) * 3)

    # Bidirectional cross-filter
    cf_col = next((c for c in ['Cross Filter Direction', 'Cross Filtering Behavior'] if c in relationships_df.columns), None)
    if cf_col:
        bidir = relationships_df[
            relationships_df[cf_col].astype(str).str.contains('Both', case=False, na=False)
        ]
        if not bidir.empty:
            print(f"⚠️  {len(bidir)} bidirectional relationship(s):")
            for _, r in bidir.head(5).iterrows():
                print(f"     • {r.get('From Table', '?')} ↔ {r.get('To Table', '?')}")
            print("\n   Bidirectional filtering can create ambiguity in DAX generation.")
            print("   ACTION: Review if these are necessary; consider alternatives.")
            issues.append(("important", "1.1 Star Schema", f"{len(bidir)} bidirectional relationship(s)"))
            score -= min(4, len(bidir) * 2)

    # Isolated visible tables
    related_tables = set()
    if 'From Table' in relationships_df.columns:
        related_tables = set(relationships_df['From Table'].dropna()) | set(relationships_df['To Table'].dropna())
    visible_tbl_names = set(visible_tables['Name'].dropna())
    isolated = visible_tbl_names - related_tables
    # Exclude parameter tables
    isolated = {t for t in isolated if not any(x in t.lower() for x in ['parameter', ' parameter', '_param'])}
    
    if isolated:
        print(f"\n⚠️  {len(isolated)} visible table(s) with no relationships:")
        for t in sorted(isolated)[:8]:
            print(f"     • {t}")
        if len(isolated) > 8:
            print(f"     ... and {len(isolated) - 8} more")
        print("\n   ACTION: Connect these tables to the model or hide them.")
        issues.append(("important", "1.1 Star Schema", f"{len(isolated)} isolated visible table(s)"))
        score -= min(6, len(isolated) * 2)

    if not issues:
        print("✅ PASSED: Relationship structure follows star schema design.")

score = max(0, score)
print(f"\n📊 Score: {score}/20")
check_scores['1.1_star_schema'] = {'achieved': score, 'max': 20, 'weight': 3}
all_issues.extend(issues)

## 1.2 ⚡ Best Practice Analyzer

**Why it matters:** BPA identifies 60+ performance, DAX quality, and error prevention issues that affect AI response time and reliability.

**What's checked:** All BPA rules (performance, DAX, formatting, maintenance)

In [ ]:
print("=" * 70)
print("CHECK 1.2: BEST PRACTICE ANALYZER")
print("=" * 70)

issues = []
score = 15  # Max points

try:
    import sempy_labs as labs
    bpa_results = labs.run_model_bpa(dataset=dataset, workspace=workspace)

    if bpa_results is not None and not bpa_results.empty:
        sev_col = 'Severity' if 'Severity' in bpa_results.columns else None
        print(f"⚠️  {len(bpa_results)} BPA recommendation(s) found:\n")
        
        if sev_col:
            sev_counts = bpa_results[sev_col].value_counts()
            for sev in ['Error', 'Warning', 'Info']:
                if sev in sev_counts.index:
                    print(f"   {sev}: {sev_counts[sev]}")
        
        # Score based on severity
        if   len(bpa_results) >= 30: score = 3
        elif len(bpa_results) >= 20: score = 6
        elif len(bpa_results) >= 10: score = 9
        elif len(bpa_results) >= 5:  score = 12
        else:                        score = 14

        print("\nTop issues (click to expand for full details):")
        display(bpa_results.head(20))
        
        issues.append(("important", "1.2 BPA", f"{len(bpa_results)} recommendations from Best Practice Analyzer"))
        
        print("\n📚 Learn more: https://learn.microsoft.com/power-bi/transform-model/service-notebooks")
    else:
        print("✅ PASSED: No BPA issues found!")

except ImportError:
    print("ℹ️  Semantic Link Labs not available for BPA.")
    print("   Install: %pip install semantic-link-labs --upgrade")
    score = 10
except Exception as e:
    print(f"⚠️  Could not run BPA: {e}")
    print("\n   Run manually: fabric.run_model_bpa(dataset=dataset, workspace=workspace)")
    score = 10

print(f"\n📊 Score: {score}/15")
check_scores['1.2_bpa'] = {'achieved': score, 'max': 15, 'weight': 3}
all_issues.extend(issues)

## 1.3 💾 Memory Analyzer

**Why it matters:** Large model size impacts Data Agent performance and workspace capacity. Optimizing memory usage improves response times.

**What's checked:** Runs `fabric.model_memory_analyzer()` to analyze model memory usage and identify optimization opportunities

**Key insights:**
- Memory footprint by table and column
- High-cardinality columns that may need optimization
- Unused or redundant columns
- Data type optimization opportunities

In [ ]:
print("=" * 70)
print("CHECK 1.3: MEMORY ANALYZER")
print("=" * 70)

print("Running Model Memory Analyzer...\n")

try:
    # Run the model memory analyzer
    memory_results = fabric.model_memory_analyzer(dataset=dataset, workspace=workspace)
    
    print("✅ Memory Analyzer completed successfully!")
    print("\n📊 Results displayed above.")
    print("\n✅ Key actions from Memory Analyzer:")
    print("   • Remove unused columns")
    print("   • Optimize high-cardinality columns")
    print("   • Review column data types")
    print("   • Consider aggregations for large fact tables")
    
    if is_direct_lake:
        print("\n💡 DIRECT LAKE SPECIFIC:")
        print("   • Run V-Order optimization on Delta tables")
        print("   • Review Direct Lake fallback scenarios")
        print("   • Monitor Direct Lake mode in Query Performance Analyzer")
        print("\n   📚 https://learn.microsoft.com/fabric/fundamentals/direct-lake-understand-storage")
    
    # Full credit for running the analyzer
    score = 5
    issues = []

except Exception as e:
    print(f"⚠️  Could not run Memory Analyzer: {e}")
    print("\n📚 Documentation:")
    print("   https://learn.microsoft.com/power-bi/transform-model/service-notebooks#model-memory-analyzer")
    print("\n🔧 Manual alternative:")
    print("   Run in a separate cell: fabric.model_memory_analyzer(dataset=dataset, workspace=workspace)")
    
    # Partial credit
    score = 3
    issues = [("important", "1.3 Memory", "Memory Analyzer could not be executed")]

check_scores['1.3_memory'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)
print(f"\n📊 Score: {score}/5")

## 1.4 🔍 Data Type Validation

**Why it matters:** Incorrect data types cause DAX errors, incorrect aggregations, and poor performance.

**What's checked:**
- Date columns stored as text
- Numeric columns stored as text
- Text columns with numeric names

In [ ]:
print("=" * 70)
print("CHECK 1.4: DATA TYPE VALIDATION")
print("=" * 70)

issues = []
score = 10

# Detect potential date columns stored as text
date_pattern = r'(date|dt_|dt$|_dt|day|month|year|quarter|fiscal|calendar|timestamp|time)'
text_date_cols = visible_columns[
    (visible_columns['Data Type'].astype(str).str.contains('string|text', case=False, na=False)) &
    (visible_columns['Column Name'].astype(str).str.lower().str.contains(date_pattern, na=False))
]

numeric_pattern = r'(amount|amt|quantity|qty|count|cnt|price|cost|rate|percent|pct|total|sum|value|val|number|num|nbr|id|key)$'
text_numeric_cols = visible_columns[
    (visible_columns['Data Type'].astype(str).str.contains('string|text', case=False, na=False)) &
    (visible_columns['Column Name'].astype(str).str.lower().str.contains(numeric_pattern, na=False))
]

if not text_date_cols.empty:
    print(f"⚠️  {len(text_date_cols)} potential date column(s) stored as text:")
    for _, row in text_date_cols.head(10).iterrows():
        print(f"     • {row.get('Table Name')}[{row.get('Column Name')}]")
    if len(text_date_cols) > 10:
        print(f"     ... and {len(text_date_cols) - 10} more")
    print("\n   ACTION: Convert to Date or DateTime type in Power Query.")
    issues.append(("important", "1.4 Data Types", f"{len(text_date_cols)} date-like columns stored as text"))
    score -= min(5, len(text_date_cols))

if not text_numeric_cols.empty:
    print(f"\n⚠️  {len(text_numeric_cols)} potential numeric column(s) stored as text:")
    for _, row in text_numeric_cols.head(10).iterrows():
        print(f"     • {row.get('Table Name')}[{row.get('Column Name')}]")
    if len(text_numeric_cols) > 10:
        print(f"     ... and {len(text_numeric_cols) - 10} more")
    print("\n   ACTION: Verify these are truly text; convert if numeric.")
    issues.append(("recommended", "1.4 Data Types", f"{len(text_numeric_cols)} numeric-like columns stored as text"))
    score -= min(3, len(text_numeric_cols))

if not issues:
    print("✅ PASSED: No obvious data type issues detected.")

score = max(0, score)
print(f"\n📊 Score: {score}/10")
check_scores['1.4_data_types'] = {'achieved': score, 'max': 10, 'weight': 2}
all_issues.extend(issues)

## 1.5 🏷️ Business-Friendly Naming

**Why it matters:** AI interprets object names literally. Technical names like `DIM_CUST_01` or `F_SLS_AMT` provide no context for natural language understanding.

**What's checked:** Database-style prefixes/suffixes, abbreviations, ALL_CAPS, cryptic names

In [ ]:
import re
import pandas as pd

# -------------------------------------------------------------------------
# CONFIGURATION: PATTERNS FOR COPILOT READINESS
# -------------------------------------------------------------------------

# 1. Technical/Database Naming Patterns
TECH_PATTERNS = [
    (r'^(DIM|FACT|FCT|STG|SRC|TBL|VW|RPT|TMP|TEMP|LKP|REF|BRG|BRIDGE|MAP|INT|ODS|SLV|GLD|SILVER|BRONZE|RAW)_', "Database prefix (DIM_, FACT_)"),
    (r'_(DIM|FACT|FCT|TBL|LKP|REF|SK|NK|AK|BK|PK|FK)$', "Database suffix (_DIM, _SK)"),
    (r'_(AMT|QTY|CNT|CT|NUM|NBR|NO|DT|TS|FLG|FLAG|IND|CD|CODE|KEY|ID|VAL)$', "Abbreviation suffix (_AMT, _ID)"),
    (r'^[A-Z][A-Z0-9_]{2,}$', "ALL_CAPS or database style"),
    (r'^[A-Za-z0-9]{1,2}$', "Very short name (1-2 chars)"),
    (r'\d{2,}$', "Ends with numbers (Table01)"),
]

# 2. Natural Language (NLP) Patterns - Critical for Copilot
NLP_PATTERNS = [
    (r'^[a-z]', "Starts with lowercase"),
    (r'[a-z][A-Z]', "CamelCase detected (use spaces)"),
    (r'_', "Snake_case / underscores detected"),
    (r'[#@$%^&*()\-+=]', "Special characters/symbols"),
    (r'\b(YoY|MoM|QoQ|YTD|MTD|LY|PY|CY)\b', "Vague acronym (use 'Year Over Year')"),
]

def is_unfriendly(name):
    # Check Technical Patterns
    for pattern, reason in TECH_PATTERNS:
        if re.search(pattern, str(name)):
            return True, reason
    # Check NLP Patterns
    for pattern, reason in NLP_PATTERNS:
        if re.search(pattern, str(name)):
            return True, reason
    return False, None

def is_redundant(table_name, field_name):
    """Flags if 'Customer' table contains 'Customer Name'."""
    t = str(table_name).lower()
    f = str(field_name).lower()
    if f.startswith(t) and f != t:
        return True, f"Redundant table prefix in '{field_name}'"
    return False, None

# -------------------------------------------------------------------------
# EXECUTION: AUDIT LOGIC
# -------------------------------------------------------------------------

print("=" * 70)
print("CHECK 1.5: COPILOT-READY BUSINESS NAMING & SEMANTICS")
print("=" * 70)

issues = []
score = 15
flagged = {'tables': [], 'columns': [], 'measures': [], 'descriptions': []}

# Audit Tables
for _, row in visible_tables.iterrows():
    name = row['Name']
    ok, reason = is_unfriendly(name)
    if ok:
        flagged['tables'].append({'Name': name, 'Reason': reason})

# Audit Columns
for _, row in visible_columns.iterrows():
    t_name = row.get('Table Name', '')
    c_name = row.get('Column Name', '')
    desc = row.get('Description', '')

    # 1. Check Unfriendly Naming
    ok, reason = is_unfriendly(c_name)
    
    # 2. Check Redundancy
    if not ok:
        ok, reason = is_redundant(t_name, c_name)
    
    if ok:
        flagged['columns'].append({'Table': t_name, 'Column': c_name, 'Reason': reason})
    
    # 3. Check for Missing Descriptions (Copilot Hints)
    if pd.isna(desc) or str(desc).strip() == "":
        flagged['descriptions'].append({'Table': t_name, 'Object': c_name, 'Type': 'Column'})

# Audit Measures
for _, row in visible_measures.iterrows():
    t_name = row.get('Table Name', '')
    m_name = row.get('Measure Name', '')
    desc = row.get('Description', '')

    ok, reason = is_unfriendly(m_name)
    if ok:
        flagged['measures'].append({'Table': t_name, 'Measure': m_name, 'Reason': reason})
    
    if pd.isna(desc) or str(desc).strip() == "":
        flagged['descriptions'].append({'Table': t_name, 'Object': m_name, 'Type': 'Measure'})

# -------------------------------------------------------------------------
# REPORTING & SCORING
# -------------------------------------------------------------------------

total_flagged = len(flagged['tables']) + len(flagged['columns']) + len(flagged['measures'])
total_objects = len(visible_tables) + len(visible_columns) + len(visible_measures)

if total_flagged == 0:
    print("✅ PASSED: All visible names are Copilot-friendly.")
else:
    # Penalize score based on percentage of bad names
    penalty = (total_flagged / max(total_objects, 1)) * 20 
    score = max(0, int(15 - penalty))

    if flagged['tables']:
        print(f"⚠️  {len(flagged['tables'])} table(s) with technical/NLP issues:")
        for item in flagged['tables'][:5]:
            print(f"     • {item['Name']}  →  {item['Reason']}")

    if flagged['columns']:
        print(f"\n⚠️  {len(flagged['columns'])} column(s) with technical/NLP issues:")
        for item in flagged['columns'][:10]:
            print(f"     • {item['Table']}[{item['Column']}]  →  {item['Reason']}")

    if flagged['measures']:
        print(f"\n⚠️  {len(flagged['measures'])} measure(s) with technical/NLP issues:")
        for item in flagged['measures'][:10]:
            print(f"     • {item['Table']}[{item['Measure']}]  →  {item['Reason']}")

    if flagged['descriptions']:
        print(f"\nℹ️  {len(flagged['descriptions'])} objects missing Descriptions (Copilot context):")
        print(f"     • Example: {flagged['descriptions'][0]['Table']}[{flagged['descriptions'][0]['Object']}]")

    print("\n💡 COPILOT TIP: Use spaces instead of underscores or CamelCase.")
    print("   Instead of 'Total_Sales' or 'TotalSales', use 'Total Sales'.")
    issues.append(("warning", "1.5 Naming", f"{total_flagged} names need NLP optimization"))

print(f"\n📊 Copilot Readiness Score: {score}/15")

## 1.6 📝 Descriptions Coverage

**Why it matters:** Descriptions are CRITICAL for AI accuracy. Every object in your AI Data Schema MUST have a clear, concise description for Copilot and AI features to understand it.

**What's checked:** Description coverage for visible tables, columns, and measures

In [ ]:
# =========================================================================
# CHECK 1.6: DESCRIPTIONS COVERAGE (ORIGINAL CODE)
# =========================================================================
print("=" * 70)
print("CHECK 1.6: DESCRIPTIONS COVERAGE")
print("=" * 70)

issues = []
score = 20  # Highest weight — most critical

tables_no_desc   = visible_tables[missing_desc_mask(visible_tables)]
cols_no_desc     = visible_columns[missing_desc_mask(visible_columns)]
measures_no_desc = visible_measures[missing_desc_mask(visible_measures)]

tbl_cov = 1 - len(tables_no_desc)   / max(len(visible_tables),   1)
col_cov = 1 - len(cols_no_desc)     / max(len(visible_columns),  1)
msr_cov = 1 - len(measures_no_desc) / max(len(visible_measures), 1)

print("Description Coverage:\n")
print(f"  Tables   : {len(visible_tables) - len(tables_no_desc):>4}/{len(visible_tables):<4} ({tbl_cov*100:>5.1f}%)")
print(f"  Columns  : {len(visible_columns) - len(cols_no_desc):>4}/{len(visible_columns):<4} ({col_cov*100:>5.1f}%)")
print(f"  Measures : {len(visible_measures) - len(measures_no_desc):>4}/{len(visible_measures):<4} ({msr_cov*100:>5.1f}%)")
print()

total_missing = len(tables_no_desc) + len(cols_no_desc) + len(measures_no_desc)

if total_missing == 0:
    print("✅ PASSED: All visible objects have descriptions.")
else:
    overall_cov = 1 - total_missing / max(len(visible_tables) + len(visible_columns) + len(visible_measures), 1)
    score = max(0, int(20 * overall_cov))

    if not tables_no_desc.empty:
        print(f"❌ {len(tables_no_desc)} table(s) missing descriptions:")
        for name in tables_no_desc['Name'].tolist()[:8]:
            print(f"     • {name}")
        if len(tables_no_desc) > 8:
            print(f"     ... and {len(tables_no_desc) - 8} more")

    if not cols_no_desc.empty:
        print(f"\n⚠️  {len(cols_no_desc)} column(s) missing descriptions (first 15):")
        for _, row in cols_no_desc.head(15).iterrows():
            print(f"     • {row.get('Table Name', '?')}[{row.get('Column Name', '?')}]")
        if len(cols_no_desc) > 15:
            print(f"     ... and {len(cols_no_desc) - 15} more")

    if not measures_no_desc.empty:
        print(f"\n⚠️  {len(measures_no_desc)} measure(s) missing descriptions (first 15):")
        for _, row in measures_no_desc.head(15).iterrows():
            print(f"     • {row.get('Table Name', '?')}[{row.get('Measure Name', '?')}]")
        if len(measures_no_desc) > 15:
            print(f"     ... and {len(measures_no_desc) - 15} more")

    print("\n🚨 CRITICAL: Add descriptions to ALL objects in your AI Data Schema before deployment.")
    issues.append(("critical", "1.6 Coverage", f"{total_missing} visible objects missing descriptions"))

print(f"\n📊 Coverage Score: {score}/20")
check_scores['1.6_descriptions'] = {'achieved': score, 'max': 20, 'weight': 3}


# =========================================================================
# CHECK 1.6.1: DESCRIPTIONS QUALITY (INFORMATIONAL AUDIT)
# =========================================================================
print("\n" + "=" * 70)
print("CHECK 1.6.1: DESCRIPTIONS QUALITY AUDIT")
print("=" * 70)

quality_issues = []

# Filter for objects that ALREADY have descriptions to audit their content
tables_with_desc   = visible_tables[~missing_desc_mask(visible_tables)]
cols_with_desc     = visible_columns[~missing_desc_mask(visible_columns)]
measures_with_desc = visible_measures[~missing_desc_mask(visible_measures)]

def check_quality(name, desc):
    n = str(name).lower().strip()
    d = str(desc).lower().strip()
    
    if d == n: 
        return "Redundant: Matches name exactly"
    if len(d) < 15: 
        return "Too brief: Less than 15 characters"
    if any(p in d for p in ['tbd', 'todo', 'placeholder', 'n/a', 'test']): 
        return "Placeholder text detected"
    if d.startswith("this column") or d.startswith("this measure"): 
        return "Circular: Starts with 'This column/measure...'"
    return None

# Audit Tables Quality
for _, row in tables_with_desc.iterrows():
    issue = check_quality(row['Name'], row['Description'])
    if issue: quality_issues.append(f"Table: {row['Name']} → {issue}")

# Audit Columns Quality
for _, row in cols_with_desc.iterrows():
    issue = check_quality(row['Column Name'], row['Description'])
    if issue: quality_issues.append(f"Column: {row['Table Name']}[{row['Column Name']}] → {issue}")

# Audit Measures Quality
for _, row in measures_with_desc.iterrows():
    issue = check_quality(row['Measure Name'], row['Description'])
    if issue: quality_issues.append(f"Measure: {row['Table Name']}[{row['Measure Name']}] → {issue}")

if not quality_issues:
    print("✅ PASSED: Existing descriptions meet AI quality standards.")
else:
    print(f"ℹ️  SUGGESTIONS: {len(quality_issues)} descriptions could be improved for Copilot:")
    for item in quality_issues[:10]:
        print(f"     • {item}")
    if len(quality_issues) > 10:
        print(f"     ... and {len(quality_issues) - 10} more")

    print("\n💡 AI TIP: Use descriptions to provide business context that the table or column name doesn't explain.")

# Final merge of issues found in 1.6 (1.6.1 is informational, so it doesn't append to all_issues)
all_issues.extend(issues)

## 1.7 🔤 Synonyms / Linguistic Schema

**Why it matters:** Synonyms help AI match natural language variations (revenue/income/sales → Total Revenue).

**What's checked:** Synonym configuration on visible objects

In [ ]:
print("=" * 70)
print("CHECK 1.7: SYNONYMS / LINGUISTIC SCHEMA")
print("=" * 70)

issues = []
score = 5

try:
    import sempy_labs as labs
    
    # Use Semantic Link Labs to list all synonyms
    df_synonyms = labs.list_synonyms(dataset=dataset, workspace=workspace)
    
    if df_synonyms is not None and not df_synonyms.empty:
        # Count synonyms by object type
        syn_by_type = df_synonyms.groupby('Object Type').size()
        
        print(f"✅ Synonyms configured:\n")
        for obj_type, count in syn_by_type.items():
            print(f"   {obj_type:<10} : {count:>4} object(s)")
        
        print(f"\n   Total: {len(df_synonyms)} synonym(s) across {len(df_synonyms['Object Name'].unique())} object(s)")
        
        # Check coverage of visible measures
        if not measures_df.empty:
            measures_with_synonyms = df_synonyms[df_synonyms['Object Type'] == 'Measure']['Object Name'].unique()
            vis_msr_names = visible_measures['Measure Name'].tolist()
            msrs_without = [m for m in vis_msr_names if m not in measures_with_synonyms]
            
            if msrs_without:
                pct_covered = 1 - len(msrs_without) / max(len(vis_msr_names), 1)
                score = max(2, int(5 * pct_covered))
                print(f"\n⚠️  {len(msrs_without)} visible measure(s) without synonyms (showing first 10):")
                for m in msrs_without[:10]:
                    print(f"     • {m}")
                if len(msrs_without) > 10:
                    print(f"     ... and {len(msrs_without) - 10} more")
                issues.append(("recommended", "1.7 Synonyms", f"{len(msrs_without)} measures without synonyms"))
            else:
                print("\n✅ All visible measures have synonyms!")
        
        print("\n📊 Synonym details:")
        display(df_synonyms.head(20))
    else:
        print("⚠️  No synonyms configured on any object.")
        print("\n   ACTION: Add synonyms for key measures and dimensions.")
        score = 1
        issues.append(("recommended", "1.7 Synonyms", "No synonyms configured"))

    print("\n💡 Synonym examples:")
    print("   • Total Revenue → synonyms: sales, income, turnover, proceeds")
    print("   • Customer → synonyms: client, account, buyer")
    print("   • YTD Revenue → synonyms: year to date revenue, revenue ytd")
    print("\n📚 Learn more:")
    print("   https://learn.microsoft.com/power-bi/natural-language/q-and-a-best-practices")

except Exception as e:
    print(f"ℹ️  Could not retrieve synonyms: {e}")
    print("\n   Manual check: Power BI Desktop → select object → Properties pane → Synonyms")
    print("   Or manage linguistic schema YAML file for bulk updates")
    score = 3

print(f"\n📊 Score: {score}/5")
check_scores['1.7_synonyms'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)

## 1.8 🏷️ Row Labels

**Why it matters:** Row labels define the primary descriptor for dimension tables, helping AI identify which column to display (e.g., "Customer Name" instead of "Customer ID").

**What's checked:** Row label configuration on dimension tables

In [ ]:
print("=" * 70)
print("CHECK 1.8: ROW LABELS FOR DIMENSION TABLES")
print("=" * 70)

issues = []
score = 5

try:
    # Using Semantic Link Labs TOM (Tabular Object Model) access
    # Row labels are exposed through Table.RowLabel TOM property
    # This property is fully represented in TMDL and accessible via XMLA
    from sempy_labs.tom import connect_semantic_model

    tables_with_row_labels = []
    tables_without = []
    all_user_tables = []

    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        for table in tom.model.Tables:
            # Skip auto-generated date tables and hidden tables
            if table.Name.startswith(('DateTableTemplate_', 'LocalDateTable_')) or table.IsHidden:
                continue
            
            all_user_tables.append(table.Name)
            
            # Column.IsDefaultLabel corresponds to Table.RowLabel TOM property
            # When using PBIP format, this is stored in TMDL and version-controlled
            row_label_column = next((col.Name for col in table.Columns if col.IsDefaultLabel and not col.IsHidden), None)
            
            if row_label_column:
                tables_with_row_labels.append((table.Name, row_label_column))
            else:
                tables_without.append(table.Name)

    print(f"📊 Total user tables analyzed: {len(all_user_tables)}")
    print()
    
    if tables_with_row_labels:
        print(f"✅ {len(tables_with_row_labels)} table(s) have row labels configured:")
        for tbl, col in tables_with_row_labels[:8]:
            print(f"     • {tbl} → [{col}]")
        if len(tables_with_row_labels) > 8:
            print(f"     ... and {len(tables_with_row_labels) - 8} more")
        print()

    if tables_without:
        print(f"⚠️  {len(tables_without)} table(s) do NOT have row labels set:")
        for tbl in tables_without[:10]:
            print(f"     • {tbl}")
        if len(tables_without) > 10:
            print(f"     ... and {len(tables_without) - 10} more")
        
        print("\n   📍 WHY THIS MATTERS:")
        print("      Row labels tell Copilot which column to display for each table.")
        print("      Without row labels, the AI may choose the wrong column (e.g., ID instead of Name).")
        print("\n   🎯 RECOMMENDATION:")
        print("      Set row labels on DIMENSION tables (Customer, Product, Store, Employee, etc.)")
        print("      For fact tables (Sales, Orders, Transactions), row labels are typically not needed.")
        print("\n   ⚙️  HOW TO SET:")
        print("      Power BI Desktop → Model view → select table → Advanced properties:")
        print("         1. Row label: Choose the column that describes each row (e.g., 'Customer Name')")
        print("         2. Key column: Choose the column that uniquely identifies each row (e.g., 'CustomerID')")
        print("\n   💡 TIP: Both Row Label AND Key Column should be set on dimension tables for best results.")
        print("\n   💾 Storage: Row labels are stored in TMDL (when using PBIP format)")
        print("      and are version-controlled via Git. Works for Import, DirectQuery, and Direct Lake.")
        
        issues.append(("important", "1.8 Row Labels", f"{len(tables_without)} tables without row labels"))
        score = max(1, 5 - min(len(tables_without) // 2, 4))
    else:
        print("✅ PASSED: All tables have row labels configured.")

    print("\n📚 Learn more:")
    print("   https://learn.microsoft.com/power-bi/natural-language/q-and-a-tooling-intro#set-a-row-label")
    print("   https://blog.crossjoin.co.uk/2024/08/25/tuning-power-bi-copilot-with-row-labels-and-key-columns/")

except Exception as e:
    print(f"ℹ️  Could not inspect row labels: {e}")
    print("\n   Manual check: Power BI Desktop → Model view → select table → Advanced → Row label")
    score = 3

print(f"\n📊 Score: {score}/5")
check_scores['1.8_row_labels'] = {'achieved': score, 'max': 5, 'weight': 2}
all_issues.extend(issues)

## 1.9 🔢 Explicit Measures (No Implicit Aggregations)

**Why it matters:** Numeric columns with default summarization create implicit measures that produce unpredictable results.

**What's checked:** Numeric columns with summarization ≠ "None"

In [ ]:
print("=" * 70)
print("CHECK 1.9: EXPLICIT MEASURES (NO IMPLICIT AGGREGATIONS)")
print("=" * 70)

issues = []
score = 15

NUMERIC_TYPES = ['Int64', 'Double', 'Decimal', 'Currency', 'Single',
                 'Whole number', 'Decimal number', 'Fixed decimal number']
SAFE_SUMMARIZE = {'None', 'DoNotSummarize', 'none', 'Do Not Summarize', 'donotsummarize'}

if 'Summarize By' not in columns_df.columns:
    print("ℹ️  'Summarize By' column not found — skipping check.")
    print("\n   Manual check: Power BI Desktop → Column tools → Summarization")
    score = 10
else:
    num_cols = visible_columns[visible_columns['Data Type'].isin(NUMERIC_TYPES)]
    implicit = num_cols[
        ~num_cols['Summarize By'].astype(str).isin(SAFE_SUMMARIZE) &
        num_cols['Summarize By'].notna()
    ]

    if implicit.empty:
        print("✅ PASSED: All numeric columns have Summarize By = None.")
    else:
        pct = len(implicit) / max(len(num_cols), 1) * 100
        score = max(0, int(15 - pct / 5))

        print(f"❌ {len(implicit)} numeric column(s) with implicit summarization:\n")
        for tbl, grp in implicit.groupby('Table Name'):
            print(f"   {tbl}:")
            for _, row in grp.iterrows():
                print(f"      • {row.get('Column Name')} (Type: {row.get('Data Type')}, Summarize: {row.get('Summarize By')})")

        print("\n🚨 CRITICAL: Set all numeric columns to 'Don't summarize'.")
        print("   Then create explicit DAX measures for calculations.")
        print("\n   📍 Power BI Desktop → Column tools → Summarization → Don't summarize")
        issues.append(("critical", "1.9 Implicit Measures", f"{len(implicit)} columns with implicit summarization"))

print(f"\n📊 Score: {score}/15")
check_scores['1.9_implicit_measures'] = {'achieved': score, 'max': 15, 'weight': 3}
all_issues.extend(issues)

## 1.10 📊 Report-Scoped Measures

**Why it matters:** Report-scoped measures are NOT accessible to AI features (Copilot, Data Agents). Migrate them to the semantic model to make them available.

**What's checked:** Detection via TOM (if available)

In [ ]:
print("=" * 70)
print("CHECK 1.10: REPORT-SCOPED MEASURES")
print("=" * 70)

print("ℹ️  This check requires access to the .pbix file or TOM model.\n")
print("🔍 Manual verification:")
print("   1. Open your reports in Power BI Desktop")
print("   2. Check Data pane for measures with a 📊 icon (report-scoped)")
print("   3. Move these to the semantic model if Data Agent should use them")
print("\n💡 TIP: Report-scoped measures are NOT visible to Data Agent.")
print("   Create them in the model instead.")

# Partial credit for awareness
score = 5
check_scores['1.10_report_measures'] = {'achieved': score, 'max': 5, 'weight': 1}
print(f"\n📊 Score: {score}/5 (manual verification required)")

## 1.12 📅 Ambiguous Date Fields

**Why it matters:** Multiple date columns without clear guidance confuse the AI when users ask time-based questions.

**What's checked:**
- Count of visible date columns
- Whether date columns have descriptions
- Risk level based on number of date columns

**Remediation:** Use AI Instructions to specify default date field, hide non-primary date columns, or add clear descriptions.

In [ ]:
print("=" * 70)
print("CHECK 1.12: AMBIGUOUS DATE FIELDS")
print("=" * 70)

issues = []
score = 5

DATE_TYPES = ['DateTime', 'Date', 'DateTimeOffset']

date_cols = visible_columns[
    visible_columns['Data Type'].isin(DATE_TYPES) |
    visible_columns['Data Type'].astype(str).str.lower().str.contains('date', na=False)
]

def desc_status(row):
    d = row.get('Description', '')
    return "has description" if d and str(d).strip() else "NO description"

if date_cols.empty:
    print("✅ OK: No visible date columns found.")
    print("   Date columns may be hidden (role-playing dimension design).")
elif len(date_cols) == 1:
    r = date_cols.iloc[0]
    print(f"✅ PASSED: Single visible date column:")
    print(f"   {r.get('Table Name')}[{r.get('Column Name')}] — {desc_status(r)}")
elif len(date_cols) <= 3:
    print(f"ℹ️  INFO: {len(date_cols)} date columns found (low ambiguity risk):")
    for _, row in date_cols.iterrows():
        print(f"   • {row.get('Table Name')}[{row.get('Column Name')}] — {desc_status(row)}")
    print("\n💡 Consider adding AI Instructions to specify default date for common questions.")
    score = 4
else:
    print(f"⚠️  WARNING: {len(date_cols)} visible date column(s) — HIGH ambiguity risk!\n")
    for tbl, grp in date_cols.groupby('Table Name'):
        print(f"   Table: {tbl}")
        for _, row in grp.iterrows():
            print(f"      • {row.get('Column Name')} ({desc_status(row)})")

    score = max(0, 5 - max(0, len(date_cols) - 3))
    issues.append(("important", "1.12 Date Fields", f"{len(date_cols)} visible date columns without clear AI guidance"))

    print("\n📍 ACTION:")
    print("   1. Hide date columns users should not query directly")
    print("   2. Add descriptions clarifying each date column's purpose")
    print("   3. Prep for AI → AI Instructions:")
    print("      Example: 'When the user asks about dates, use [Order Date] by default'")
    print("   4. Create Verified Answers for most common time-based questions")

print(f"\n📊 Score: {score}/5")
check_scores['1.12_date_fields'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)

## 1.13 👁️ Hidden Objects Risk

**Why it matters:**
- Verified Answers break silently if they reference hidden columns
- Hidden columns without descriptions cannot be understood by downstream processes

**What's checked:**
- Count of hidden objects
- Hidden columns lacking descriptions
- Verified Answers risk assessment

**Remediation:** Add descriptions to all hidden columns, especially those potentially referenced in Verified Answers.

In [ ]:
print("=" * 70)
print("CHECK 1.13: HIDDEN OBJECTS RISK")
print("=" * 70)

issues = []
score = 5

hidden_tables   = tables_df[is_hidden_mask(tables_df)]
hidden_columns  = columns_df[is_hidden_mask(columns_df)]
hidden_measures = measures_df[is_hidden_mask(measures_df)]

print(f"   Hidden objects summary:")
print(f"      Tables   : {len(hidden_tables):>4}")
print(f"      Columns  : {len(hidden_columns):>4}")
print(f"      Measures : {len(hidden_measures):>4}")
print()

# Hidden columns without descriptions — double risk
if not hidden_columns.empty:
    hc_no_desc = hidden_columns[
        hidden_columns['Description'].isna() | (hidden_columns['Description'].astype(str).str.strip() == '')
    ] if 'Description' in hidden_columns.columns else hidden_columns

    pct = len(hc_no_desc) / max(len(hidden_columns), 1) * 100

    if pct > 50:
        print(f"⚠️  WARNING: {len(hc_no_desc)} hidden column(s) lack descriptions ({pct:.0f}% of hidden columns).")
        print("   If any of these are referenced by Verified Answers, the answer will silently fail.")
        score = 3
        issues.append(("important", "1.13 Hidden Objects", f"{len(hc_no_desc)} hidden columns without descriptions — Verified Answers risk"))
    else:
        print(f"✅ PASSED: Most hidden columns have descriptions ({100-pct:.0f}% coverage).")
else:
    print("✅ PASSED: No hidden columns found.")

print("\n🚨 REMINDER: Verified Answers will NOT work if they reference hidden columns.")
print("   Ensure all columns referenced in Verified Answers are VISIBLE in the model.")

print(f"\n📊 Score: {score}/5")
check_scores['1.13_hidden_objects'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)

## 1.14 📦 Model Complexity / Bloat

**Why it matters:** Large models with unnecessary objects create noise for the DAX generation tool, reducing accuracy and increasing response latency.

**What's checked:**
- Visible column count (threshold: >500)
- Visible measure count (threshold: >150)
- Visible helper/intermediate measures

**Remediation:** Configure focused AI Data Schema, hide helper measures, remove unnecessary objects.

In [ ]:
print("=" * 70)
print("CHECK 1.14: MODEL COMPLEXITY / BLOAT")
print("=" * 70)

issues = []
score = 5

vis_tbl_n = len(visible_tables)
vis_col_n = len(visible_columns)
vis_msr_n = len(visible_measures)

print(f"   Model size:")
print(f"      Visible tables   : {vis_tbl_n:>4}")
print(f"      Visible columns  : {vis_col_n:>4}")
print(f"      Visible measures : {vis_msr_n:>4}")
print()

# Detect helper measures that are visible (they should be hidden)
HELPER_PATTERN = r'\b(helper|_h$|aux|auxiliary|temp|tmp|working|intermediate|_int$|__[a-z])\b'
visible_helpers = visible_measures[
    visible_measures['Measure Name'].astype(str).str.lower().str.contains(HELPER_PATTERN, regex=True, na=False)
]

if not visible_helpers.empty:
    print(f"⚠️  WARNING: {len(visible_helpers)} potentially visible helper/intermediate measure(s):")
    for _, row in visible_helpers.head(10).iterrows():
        print(f"      • {row.get('Table Name')}[{row.get('Measure Name')}]")
    if len(visible_helpers) > 10:
        print(f"      ... and {len(visible_helpers) - 10} more")
    print("   Helper measures should be hidden or excluded from your AI Data Schema.")
    issues.append(("important", "1.14 Complexity", f"{len(visible_helpers)} visible helper/intermediate measure(s)"))
    score -= 2

if vis_col_n > 500:
    print(f"\n⚠️  WARNING: {vis_col_n} visible columns — high noise risk for the AI.")
    print("   Use Prep for AI → AI Data Schema to define a focused subset.")
    issues.append(("recommended", "1.14 Complexity", f"High visible column count ({vis_col_n}) — configure AI Data Schema"))
    score -= 1

if vis_msr_n > 150:
    print(f"\n⚠️  WARNING: {vis_msr_n} visible measures — review for redundant intermediate measures.")
    print("   Hide intermediate measures used only as building blocks for other measures.")
    issues.append(("recommended", "1.14 Complexity", f"High visible measure count ({vis_msr_n}) — review for redundancy"))
    score -= 1

if not issues:
    print("✅ PASSED: Model complexity is manageable for AI use.")

print("\n💡 TIP: Use Best Practice Analyzer and Memory Analyzer to identify")
print("   unnecessary columns, high-cardinality columns, and inefficient DAX patterns.")

score = max(0, score)
print(f"\n📊 Score: {score}/5")
check_scores['1.14_model_bloat'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)

## 1.17 ⚡ Performance Analyzer for AI Data Schema Measures

**Why it matters:** Slow-executing measures in your AI Data Schema cause timeout errors and poor user experience. You must validate performance specifically for the measures selected in Prep for AI.

**What's checked:** Manual — run Performance Analyzer in Power BI Desktop targeting only measures included in AI Data Schema

**Key insights:**
- Measures exceeding 2 seconds warrant optimization before deployment
- Storage Engine vs Formula Engine query ratio
- Direct Lake fallback scenarios that trigger Import mode scans

In [ ]:
print("=" * 70)
print("CHECK 1.17: PERFORMANCE ANALYZER FOR AI DATA SCHEMA MEASURES")
print("=" * 70)

print("🔵 MANUAL CHECK — requires Power BI Desktop.\n")
print("📍 HOW TO RUN:")
print("   Power BI Desktop → View tab → Performance Analyzer → Start recording")
print("   Then create visuals using each measure in your AI Data Schema.\n")
print("✅ CHECKLIST:")
print()
print("   [ ] 1. Open report connected to this semantic model in Power BI Desktop")
print("   [ ] 2. Enable Performance Analyzer (View → Performance Analyzer)")
print("   [ ] 3. Click 'Start recording'")
print("   [ ] 4. For each measure in AI Data Schema, create a simple visual using it")
print("   [ ] 5. Review DAX query times — target < 2 seconds per measure")
print("   [ ] 6. Export results: click '...' → 'Export data'")
print()
print("⚠️  WARNING THRESHOLDS:")
print("   • < 500ms  ✅ Excellent — no action needed")
print("   • 500ms–2s ⚠️  Acceptable — monitor under production load")
print("   • 2s–5s    🟠 Slow — optimize before deployment")
print("   • > 5s     🔴 Critical — will cause Data Agent timeouts")
print()
print("💡 OPTIMIZATION ACTIONS:")
print("   • Run BPA (Check 1.2) to identify inefficient DAX patterns")
print("   • Remove column store scans with SUMMARIZECOLUMNS rewrites")
print("   • Use aggregation tables for large fact tables")
print("   • Review Direct Lake fallback if using Direct Lake mode (Check 1.3)")
print("   • Check if measure uses CALCULATE with complex filter contexts")
print()
print("📚 Documentation:")
print("   https://learn.microsoft.com/power-bi/create-reports/desktop-performance-analyzer")

score = 5 if prep_for_ai_configured else 2
check_scores['1.17_perf_analyzer'] = {'achieved': score, 'max': 5, 'weight': 1}
if not prep_for_ai_configured:
    all_issues.append(("important", "1.17 Perf Analyzer", "Run Performance Analyzer against AI Data Schema measures before deployment"))
print(f"\n📊 Score: {score}/5 (manual verification required)")

## 1.18 🔁 Duplicate Column Names Across Tables

**Why it matters:** When two tables expose a column with the same name (e.g., `Sales[Date]` and `Orders[Date]`), the AI's DAX generator can pick the wrong table, leading to incorrect filter context and wrong results without any obvious error.

**What's checked:** Automated — scans all visible columns for names appearing in more than one table

In [ ]:
print("=" * 70)
print("CHECK 1.18: DUPLICATE COLUMN NAMES ACROSS TABLES")
print("=" * 70)

issues = []
score = 5

if 'Column Name' in visible_columns.columns and 'Table Name' in visible_columns.columns:
    # Find column names that appear in more than one table
    col_table_counts = (
        visible_columns
        .groupby('Column Name')['Table Name']
        .agg(list)
        .reset_index()
    )
    col_table_counts.columns = ['Column Name', 'Tables']
    col_table_counts['Table Count'] = col_table_counts['Tables'].apply(len)

    duplicates = col_table_counts[col_table_counts['Table Count'] > 1].sort_values('Table Count', ascending=False)

    if duplicates.empty:
        print("✅ PASSED: No duplicate column names found across tables.")
    else:
        print(f"⚠️  {len(duplicates)} column name(s) appear in multiple tables:\n")
        for _, row in duplicates.head(20).iterrows():
            tables_str = ', '.join(row['Tables'][:5])
            if len(row['Tables']) > 5:
                tables_str += f" ... +{len(row['Tables']) - 5} more"
            print(f"     • [{row['Column Name']}]  →  {row['Table Count']} tables: {tables_str}")

        if len(duplicates) > 20:
            print(f"\n     ... and {len(duplicates) - 20} more duplicate column names")

        print("\n📍 WHY THIS CAUSES ERRORS:")
        print("   When you ask 'Filter by Date', the AI doesn't know which table's")
        print("   [Date] column to use — it may guess wrong, silently returning bad data.")
        print()
        print("✅ REMEDIATION OPTIONS:")
        print("   Option A (Preferred): Rename ambiguous columns to be table-specific")
        print("     Example: Sales[Date] → Sales[Order Date]")
        print("     Example: Orders[Date] → Orders[Delivery Date]")
        print("   Option B: Add clear descriptions to distinguish identical column names")
        print("   Option C: Hide the non-primary instance in the AI Data Schema")
        print("   Option D: Add AI Instructions to specify which table to use by default")
        print()
        print("💡 FOCUS: Prioritize columns included in your AI Data Schema.")

        penalty = min(4, len(duplicates))
        score = max(1, 5 - penalty)
        issues.append(("important", "1.18 Duplicate Columns", f"{len(duplicates)} column name(s) shared across multiple tables"))
else:
    print("ℹ️  Column metadata not available — skipping check.")

score = max(0, score)
print(f"\n📊 Score: {score}/5")
check_scores['1.18_duplicate_cols'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)

---
# 2️⃣ PREP FOR AI — AI DATA SCHEMA

This section validates the **AI Data Schema** configuration (Prep for AI → Simplify data schema).

## 2.1 🎯 AI Data Schema Selection

**Why it matters:** This is THE most important configuration. Select ONLY tables/columns/measures relevant to your users' AI-powered questions.

**What's checked:** Manual verification via checklist

In [ ]:
print("=" * 70)
print("CHECK 2.1: AI DATA SCHEMA SELECTION (MANUAL)")
print("=" * 70)

print("🚨 CRITICAL: This is a MANUAL configuration in Power BI.\n")
print("📍 Power BI Desktop or Service → Home → Prep data for AI → Simplify data schema")
print("\n✅ CHECKLIST:")
print("\n   [ ] 1. Define your AI scope (what questions should users ask?)")
print("   [ ] 2. Select ONLY relevant tables for those questions")
print("   [ ] 3. Include ALL dependent objects for selected measures (see Check 2.2)")
print("   [ ] 4. Exclude helper/intermediate measures")
print("   [ ] 5. Exclude duplicate measures (keep only canonical versions)")
print("   [ ] 6. Verify no hidden fields referenced by Verified Answers")
print("   [ ] 7. Test selected tables with Copilot (see Section 5)")
print("\n⚠️  IMPORTANT: Start narrow — it's easier to add objects than remove them.")
print("\n📚 Documentation:")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-data-schema")

score = 10 if prep_for_ai_configured else 0
check_scores['2.1_ai_data_schema'] = {'achieved': score, 'max': 20, 'weight': 3}
if not prep_for_ai_configured:
    all_issues.append(("critical", "2.1 AI Data Schema", "AI Data Schema not yet configured"))
print(f"\n📊 Score: {score}/20 (manual verification required)")

## 2.2 🔗 Measure Dependencies

**Why it matters:** If a measure depends on columns/measures not in AI Data Schema, DAX generation will fail or produce incorrect results.

**What's checked:** Measure dependencies via Semantic Link Labs

In [ ]:
print("=" * 70)
print("CHECK 2.2: MEASURE DEPENDENCIES ANALYSIS")
print("=" * 70)

try:
    from sempy_labs import get_measure_dependencies
    
    print("Retrieving measure dependencies...\n")
    deps = get_measure_dependencies(dataset=dataset, workspace=workspace)

    if deps is not None and not deps.empty:
        n_measures = deps['Measure Name'].nunique() if 'Measure Name' in deps.columns else len(deps)
        print(f"✅ Dependency data retrieved for {n_measures} measure(s).\n")
        print("📋 Sample dependencies (first 30 rows):")
        display(deps.head(30))
        
        print("\n🚨 CRITICAL ACTION:")
        print("   When configuring AI Data Schema, include ALL dependent objects for each")
        print("   measure you select. Missing dependencies cause incorrect DAX generation.")
        print("\n💡 WORKFLOW:")
        print("   1. Select measure in AI Data Schema")
        print("   2. Find its row in the dependency table above")
        print("   3. Add ALL referenced tables, columns, and measures to AI Data Schema")
        
        score = 15
        check_scores['2.2_dependencies'] = {'achieved': score, 'max': 15, 'weight': 3}
    else:
        print("ℹ️  No dependency data returned.")
        score = 10
        check_scores['2.2_dependencies'] = {'achieved': score, 'max': 15, 'weight': 3}

except ImportError:
    print("ℹ️  get_measure_dependencies not available.")
    print("\n   Install: %pip install semantic-link-labs --upgrade")
    score = 10
    check_scores['2.2_dependencies'] = {'achieved': score, 'max': 15, 'weight': 3}
except Exception as e:
    print(f"⚠️  Could not analyze dependencies: {e}")
    score = 10
    check_scores['2.2_dependencies'] = {'achieved': score, 'max': 15, 'weight': 3}

print(f"\n📊 Score: {score}/15")

## 2.3 🔧 Helper Measures Detection

**Why it matters:** Helper/intermediate measures visible in the model should be excluded from AI Data Schema to reduce noise.

**What's checked:** Visible measures with helper/intermediate naming patterns

In [ ]:
print("=" * 70)
print("CHECK 2.3: HELPER MEASURES DETECTION")
print("=" * 70)

issues = []
score = 5

HELPER_PATTERN = r'\b(helper|_h$|_helper$|aux|auxiliary|temp|tmp|working|intermediate|_int$|__[a-z]|_calc$|internal)\b'
visible_helpers = visible_measures[
    visible_measures['Measure Name'].astype(str).str.lower().str.contains(HELPER_PATTERN, regex=True, na=False)
]

if visible_helpers.empty:
    print("✅ PASSED: No obvious helper measures found in visible measures.")
else:
    print(f"⚠️  {len(visible_helpers)} potential helper/intermediate measure(s):\n")
    for _, row in visible_helpers.head(12).iterrows():
        print(f"     • {row.get('Table Name')}[{row.get('Measure Name')}]")
    if len(visible_helpers) > 12:
        print(f"     ... and {len(visible_helpers) - 12} more")
    
    print("\n📍 ACTION:")
    print("   1. Hide these measures if they're only building blocks")
    print("   2. Or exclude from AI Data Schema if they must stay visible")
    
    score = max(1, 5 - len(visible_helpers))
    issues.append(("important", "2.3 Helper Measures", f"{len(visible_helpers)} visible helper measures"))

print(f"\n📊 Score: {score}/5")
check_scores['2.3_helper_measures'] = {'achieved': score, 'max': 5, 'weight': 1}
all_issues.extend(issues)

---
# 3️⃣ PREP FOR AI — VERIFIED ANSWERS

**Critical configuration:** Create verified answers for your 5-10 most common or complex questions.

> 💡 **DevOps Insight:** Unlike Data Agent instructions (which live at service level with no Git story), Verified Answers travel with your model and benefit from full version control.

## 🔍 Where Verified Answers Live

- ✅ Are **fully honored by Data Agents** when present in the semantic model

**Verified Answers are stored in your semantic model's `linguisticMetadata` property** — the same location as AI Instructions and synonyms. This means they:- ✅ Can be reviewed in pull requests

- ✅ Are version-controlled via TMDL when using PBIP format- ✅ Flow through deployment pipelines with your model

In [ ]:
print("=" * 70)
print("CHECK 3: VERIFIED ANSWERS (MANUAL)")
print("=" * 70)

print("🚨 CRITICAL: Verified Answers stored in linguisticMetadata (version-controlled).\n")
print("📍 Power BI Desktop or Service → Home → Prep data for AI → Verified answers")
print("\n💾 STORAGE LOCATION:")
print("   • Verified Answers are stored in model's linguisticMetadata property")
print("   • When using PBIP format, they appear in TMDL files")
print("   • Fully version-controlled via Git when using source control")
print("   • Travel with the model through deployment pipelines")
print("\n✅ BEST PRACTICES CHECKLIST:")
print("\n   [ ] 1. Identify 5-10 most common questions from your team")
print("   [ ] 2. Create verified answer using appropriate visual type")
print("   [ ] 3. Add 5-7 complete, robust trigger questions (NOT partial phrases)")
print("   [ ] 4. Include both formal and conversational phrasings")
print("        Example triggers for 'YTD Revenue by Region':")
print("        - What is the year to date revenue by region?")
print("        - Show revenue YTD broken down by region")
print("        - YTD sales performance per region")
print("        - How are regions performing this year?")
print("   [ ] 5. Configure up to 3 filters for flexible slicing")
print("   [ ] 6. ⚠️  CRITICAL: Ensure ALL fields used are VISIBLE (not hidden)")
print("   [ ] 7. Test each trigger question for exact and semantic matching")
print("   [ ] 8. After renaming objects, re-save affected verified answers")
print("   [ ] 9. Review TMDL changes in pull requests (PBIP format)")
print("\n🚫 COMMON MISTAKES:")
print("   • Using partial phrases as triggers ('revenue by region')")
print("   • Referencing hidden columns → verified answer will FAIL silently")
print("   • Too few trigger questions (need 5-7 for good coverage)")
print("   • Not testing both formal and casual language")
print("   • Editing in service and losing version control")
print("\n✅ DEVOPS TIP:")
print("   Configure Verified Answers in Power BI Desktop, save as PBIP, commit to Git.")
print("   Changes to verified answers will appear in your TMDL diffs for review.")
print("\n📚 Documentation:")
print("   https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-verified-answers")

score = 10 if prep_for_ai_configured else 0
check_scores['3_verified_answers'] = {'achieved': score, 'max': 15, 'weight': 3}
if not prep_for_ai_configured:
    all_issues.append(("critical", "3 Verified Answers", "Verified Answers not yet configured"))
print(f"\n📊 Score: {score}/15 (manual verification required)")

---
# 4️⃣ PREP FOR AI — AI INSTRUCTIONS

**Critical configuration:** Define business terminology, date defaults, and metric preferences.

> 🎯 **DevOps Insight:** When AI instructions live in the semantic model, they become part of PBIP folder structure, can be reviewed in pull requests, and flow through deployment pipelines. Data Agent instructions create configuration drift risk.

## 🔍 The Two Instruction Surfaces

**Push as much instruction content as possible into the semantic model layer** (Prep for AI), where it benefits from version control. Keep Data Agent instructions minimal — focused only on cross-source routing.

There are **two distinct places** where AI instructions can be configured:

## 💡 Best Practice Pattern

### 1️⃣ Semantic Model AI Instructions (Prep for AI)

**Storage:** `linguisticMetadata.CustomInstructions` property in model metadata  **Honored by:** Data Agent only

**Access:** Power BI Desktop → Prep data for AI → Add AI instructions  **DevOps:** ⚠️ Manual tracking required, risk of configuration drift  

**Version Control:** ✅ Stored in TMDL (when using PBIP format)  **Version Control:** ❌ No TMDL representation, no Git story  

**DevOps:** ✅ Diff-able in Git, flows through CI/CD pipelines  **Access:** Fabric Portal → Data Agent → Instructions  

**Honored by:** Data Agents, Copilot in Power BI**Storage:** Service-level configuration (NOT in model metadata)  

### 2️⃣ Data Agent Instructions (Fabric Service)

In [ ]:
print("=" * 70)
print("CHECK 4: AI INSTRUCTIONS (MANUAL)")
print("=" * 70)

print("🚨 CRITICAL: AI Instructions at SEMANTIC MODEL level (version-controlled).\n")
print("📍 Power BI Desktop → Home → Prep data for AI → Add AI instructions")
print("\n💾 STORAGE & VERSION CONTROL:")
print("   • Instructions stored in linguisticMetadata.CustomInstructions")
print("   • linguisticMetadata version: 4.0.0 → 4.2.0 when instructions added")
print("   • Model annotation PBI_ProTooling includes 'CopilotTooling' flag")
print("   • When using PBIP format: fully version-controlled via TMDL")
print("   • Changes appear in Git diffs for review in pull requests")
print("\n📋 WHAT TO INCLUDE (SEMANTIC MODEL INSTRUCTIONS):")
print("\n   1️⃣  BUSINESS TERMINOLOGY")
print("      Define organization-specific terms and abbreviations:")
print("      Example: 'TMS is total media spend, calculated using [Total Media Spend]'")
print("      Example: 'Top performer = sales rep achieving 110%+ of monthly quota'")
print("\n   2️⃣  TIME PERIOD DEFINITIONS")
print("      Clarify fiscal year, seasons, reporting periods:")
print("      Example: 'Fiscal year starts July 1'")
print("      Example: 'Peak season is November through January'")
print("\n   3️⃣  DEFAULT DATE FIELD")
print("      Specify which date to use when ambiguous:")
print("      Example: 'Unless specified, use Order Date for date-based questions'")
print("\n   4️⃣  METRIC PREFERENCES")
print("      Guide which measure to use for common questions:")
print("      Example: 'For profitability questions, use Contribution Margin'")
print("      Example: 'Revenue questions should use Net Revenue after discounts'")
print("\n   5️⃣  CALCULATION GROUPS & FIELD PARAMETERS")
print("      If using these features, explain how they work:")
print("      Example: 'Time intelligence is handled by the [Time Calc] calculation group'")
print("\n   6️⃣  COMPLEX DAX PATTERNS (optional)")
print("      Provide example DAX for complex scenarios to guide AI")
print("\n   7️⃣  DEFAULT GROUPINGS & ANALYSIS PREFERENCES")
print("      Define how the AI should group and analyze data by default:")
print("      Example: 'When analyzing sales performance, default grouping is Region then Product Category'")
print("      Example: 'Comparative analysis should default to prior year as the comparison period'")
print("      Example: 'Rank analyses should show Top 10 unless the user specifies otherwise'")
print("      Example: 'Currency values should be displayed in thousands (K) unless specified'")
print()
print("   [ ] Add default grouping preference to AI Instructions")
print("   [ ] Add default comparison period (e.g., prior year, prior month)")
print("   [ ] Specify default sort order for ranked questions")
print("   [ ] Define default aggregation level (product, category, region)")
print("\n⚠️  WHY THIS MATTERS — VERSION CONTROL:")
print("   AI Instructions in your semantic model are stored in PBIP/TMDL format.")
print("   This means they're version-controlled via Git and flow through deployment pipelines.")
print("   Keep all semantic model guidance in Prep for AI, not in separate configuration files.")
print("\n🔄 DEVOPS PATTERN:")
print("   1. Configure instructions in Power BI Desktop (Prep for AI)")
print("   2. Save model as PBIP format")
print("   3. Commit TMDL files to Git")
print("   4. Deploy through pipeline (instructions travel with model)")
print("   5. For multi-source scenarios, consider using Fabric Data Agents with cross-source routing")

print("\n💡 TIP: Test responses BEFORE adding instructions to identify gaps.")print(f"\n📊 Score: {score}/15 (manual verification required)")

print("   Instructions should clarify, not contradict verified answers.")    all_issues.append(("critical", "4 AI Instructions", "AI Instructions not yet configured"))

print("\n📚 Documentation:")if not prep_for_ai_configured:

print("   https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai-instructions")check_scores['4_ai_instructions'] = {'achieved': score, 'max': 15, 'weight': 3}

print("   https://learn.microsoft.com/fabric/data-science/data-agent-semantic-model")score = 10 if prep_for_ai_configured else 0


---
# 5️⃣ COPILOT & AI TESTING

**Critical requirement:** Test your Prep for AI configuration with Power BI Copilot or Fabric Data Agents to validate accuracy.

In [ ]:
print("=" * 70)
print("CHECK 5: COPILOT & AI TESTING (MANUAL)")
print("=" * 70)

print("🚨 CRITICAL: Test your Prep for AI configuration with real user questions.\n")
print("📍 WHERE TO TEST:")
print("   • Power BI Service → Open report → Copilot pane")
print("   • Fabric Portal → Data Agent (if using Data Agents)")
print("   • Power BI Desktop → Copilot in the ribbon (preview)")
print("\n✅ TESTING CHECKLIST:")
print("\n   [ ] 1. BASELINE TESTING (before adding AI Instructions)")
print("      • Ask 10-20 common questions")
print("      • Review generated DAX in each response")
print("      • Verify data accuracy")
print("      • Identify which questions fail or return incorrect results")
print()
print("   [ ] 2. VERIFIED ANSWERS TESTING")
print("      • Test each trigger question individually")
print("      • Verify exact matches return the verified answer")
print("      • Test semantic variations")
print("      • Confirm filters work as expected")
print()
print("   [ ] 3. AI DATA SCHEMA VALIDATION")
print("      • Test questions using objects IN the AI Data Schema → should work")
print("      • Test questions using objects NOT in schema → should fail gracefully")
print()
print("   [ ] 4. PERFORMANCE TESTING")
print("      • Measure response times for complex queries")
print("      • If slow: analyze DAX, optimize model, simplify instructions")
print()
print("   [ ] 5. COPILOT FEEDBACK REVIEW")
print("      • Check Copilot thumbs up/down feedback from users")
print("      • Identify patterns in negative feedback")
print("      • Iterate on AI Instructions based on feedback")
print()
print("\n💡 FOR DATA AGENT USERS:")
print("   📍 Fabric Portal → Data Agent → Add semantic model")
print("   • Select THE SAME tables as in Prep for AI → AI Data Schema")
print("   • Test responses BEFORE adding Data Agent instructions")
print("   • Keep Data Agent instructions minimal (cross-source routing only)")
print("   • All semantic model guidance belongs in Prep for AI → AI Instructions")
print()
print("📚 Documentation:")
print("   Power BI Copilot: https://learn.microsoft.com/power-bi/create-reports/copilot-introduction")
print("   Prep for AI: https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai")
print("   Data Agents: https://learn.microsoft.com/fabric/data-science/data-agent-semantic-model")

score = 0  # Must be tested separately
check_scores['5_copilot_testing'] = {'achieved': score, 'max': 20, 'weight': 3}
all_issues.append(("critical", "5 Copilot Testing", "Test with Copilot or Data Agents before deployment"))
print(f"\n📊 Score: {score}/20 (manual testing required)")

---
# 6️⃣ TESTING & VALIDATION

**Critical phase:** Systematic testing before deployment ensures accuracy and identifies configuration gaps.

In [ ]:
print("=" * 70)
print("CHECK 6: TESTING & VALIDATION (MANUAL)")
print("=" * 70)

print("🚨 CRITICAL: Test systematically BEFORE adding AI Instructions.\n")
print("✅ TESTING WORKFLOW:")
print("\n   1️⃣  BASELINE TESTING (before AI Instructions)")
print("      • Ask 10-20 common questions")
print("      • Review DAX query in each response")
print("      • Verify data accuracy")
print("      • Identify which questions fail or return incorrect results")
print("      → Use results to guide Prep for AI configuration tweaks")
print("\n   2️⃣  VERIFIED ANSWERS TESTING")
print("      • Test each trigger question individually")
print("      • Verify exact matches return the verified answer")
print("      • Test semantic variations")
print("      • Confirm filters work as expected")
print("\n   3️⃣  AI DATA SCHEMA VALIDATION")
print("      • Test questions using objects IN the AI Data Schema → should work")
print("      • Test questions using objects NOT in schema → should fail gracefully")
print("      → Confirms schema is correctly configured")
print("\n   4️⃣  PERFORMANCE TESTING")
print("      • Measure response times for complex queries")
print("      • If slow: analyze DAX performance, optimize model, simplify instructions")
print("      • Use Performance Analyzer in Power BI Desktop")
print("\n   5️⃣  AUTOMATED EVALUATION (Python SDK)")
print("      • Create ground truth Q&A dataset")
print("      • Run batch evaluation")
print("      • Track accuracy metrics over time")
print("\n      📚 SDK: https://learn.microsoft.com/fabric/data-science/fabric-data-agent-sdk")
print("      📚 Evaluation: https://learn.microsoft.com/fabric/data-science/evaluate-data-agent")
print("\n   6️⃣  DIAGNOSTICS REVIEW")
print("      • Download diagnostics logs from failed/incorrect responses")
print("      • Identify which stage caused the issue:")
print("        - Intent recognition")
print("        - DAX generation")
print("        - Data retrieval")
print("      → Guides which configuration needs adjustment")
print("\n      📚 Diagnostics: https://learn.microsoft.com/fabric/data-science/evaluate-data-agent#diagnostics-button")
print("\n   7️⃣  ITERATION")
print("      • Based on testing results, adjust:")
print("        - AI Data Schema (add/remove objects)")
print("        - AI Instructions (clarify terminology)")
print("        - Verified Answers (add more triggers)")
print("      • Re-test after each change")
print("\n💡 DEPLOYMENT:")
print("   [ ] Use Git integration for version control")
print("   [ ] Use Deployment Pipelines for dev → test → prod")
print("   [ ] Add Data Agent description before publishing")
print("   [ ] For M365 Copilot: add publishing instructions")
print("\n      📚 Git: https://learn.microsoft.com/fabric/data-science/data-agent-git-integration")
print("      📚 M365: https://learn.microsoft.com/fabric/data-science/data-agent-microsoft-365-copilot")

score = 0  # Must be done separately
check_scores['6_testing'] = {'achieved': score, 'max': 20, 'weight': 3}
all_issues.append(("critical", "6 Testing", "Systematic testing required before deployment"))
print(f"\n📊 Score: {score}/20 (manual validation required)")

---
# 🏆 FINAL READINESS SCORECARD

In [ ]:
print()
print("=" * 75)
print("   🤖  SEMANTIC MODEL AI READINESS SCORECARD v1.0")
print(f"       Model: {dataset}  |  Workspace: {workspace}")
print("=" * 75)
print()

# Calculate weighted score
total_weighted = 0
total_max_weighted = 0

for key, data in check_scores.items():
    achieved = data['achieved']
    max_pts = data['max']
    weight = data['weight']
    total_weighted += achieved * weight
    total_max_weighted += max_pts * weight

overall_pct = (total_weighted / total_max_weighted * 100) if total_max_weighted > 0 else 0

# Check labels
CHECK_LABELS = {
    '1.1_star_schema': ('1.1 Star Schema Validation', 20, 3),
    '1.2_bpa': ('1.2 Best Practice Analyzer', 15, 3),
    '1.3_memory': ('1.3 Memory Analyzer', 5, 1),
    '1.4_data_types': ('1.4 Data Type Validation', 10, 2),
    '1.5_naming': ('1.5 Business-Friendly Naming', 15, 3),
    '1.6_descriptions': ('1.6 Descriptions Coverage', 20, 3),
    '1.7_synonyms': ('1.7 Synonyms', 5, 1),
    '1.8_row_labels': ('1.8 Row Labels', 5, 2),
    '1.9_implicit_measures': ('1.9 Explicit Measures', 15, 3),
    '1.10_report_measures': ('1.10 Report-Scoped Measures', 5, 1),
    '1.11_duplicates': ('1.11 Duplicate Measures', 10, 2),
    '1.17_perf_analyzer': ('1.17 Perf Analyzer (AI Schema)', 5, 1),
    '1.18_duplicate_cols': ('1.18 Duplicate Column Names', 5, 1),
    '2.1_ai_data_schema': ('2.1 AI Data Schema', 20, 3),
    '2.2_dependencies': ('2.2 Measure Dependencies', 15, 3),
    '2.3_helper_measures': ('2.3 Helper Measures', 5, 1),
    '3_verified_answers': ('3 Verified Answers', 15, 3),
    '4_ai_instructions': ('4 AI Instructions', 15, 3),
    '5_copilot_testing': ('5 Copilot & AI Testing', 20, 3),
    '6_testing': ('6 Testing & Validation', 20, 3),
}

print(f"  {'Check':<45} {'Score':>12}   {'Progress'}")
print("  " + "─" * 73)

for key, (label, max_pts, weight) in CHECK_LABELS.items():
    if key in check_scores:
        achieved = check_scores[key]['achieved']
        pct = achieved / max_pts * 100 if max_pts > 0 else 0
        bar_fill = int(pct / 5)
        bar = "█" * bar_fill + "░" * (20 - bar_fill)
        
        if pct >= 80:
            icon = "✅"
        elif pct >= 50:
            icon = "⚠️ "
        else:
            icon = "❌"
        
        print(f"  {icon} {label:<44} {achieved:>3}/{max_pts:<3}  {bar}")

print("  " + "─" * 73)
print(f"  {'TOTAL SCORE (weighted)':<45} {int(overall_pct):>3}%")
print()

# Rating
if   overall_pct >= 90: rating, comment = "🟢 READY FOR DEPLOYMENT", "Excellent — monitor and iterate based on user feedback"
elif overall_pct >= 75: rating, comment = "🟡 MOSTLY READY", "Good foundation — address flagged items before deployment"
elif overall_pct >= 50: rating, comment = "🟠 NEEDS WORK", "Significant improvements needed before production use"
else:                   rating, comment = "🔴 NOT READY", "Complete critical configuration before deploying"

print(f"  Rating : {rating}")
print(f"  Status : {comment}")
print()

# Prioritized issues
critical = [(s, c, d) for s, c, d in all_issues if s == 'critical']
important = [(s, c, d) for s, c, d in all_issues if s == 'important']
recommended = [(s, c, d) for s, c, d in all_issues if s == 'recommended']

if critical:
    print("  🔴 CRITICAL — Must fix before deployment:")
    for _, check, desc in critical:
        print(f"     • [{check}] {desc}")
    print()

if important:
    print("  🟠 IMPORTANT — Fix before production use:")
    for _, check, desc in important:
        print(f"     • [{check}] {desc}")
    print()

if recommended:
    print("  🟡 RECOMMENDED — Improves accuracy:")
    for _, check, desc in recommended:
        print(f"     • [{check}] {desc}")
    print()

if not (critical or important or recommended):
    print("  ✅ No automated issues detected — complete manual checks above!")

print()
print("=" * 75)
print("  📚 KEY RESOURCES")
print("=" * 75)
print("  Power BI Copilot:")
print("    https://learn.microsoft.com/power-bi/create-reports/copilot-introduction")
print("  Prep for AI Documentation:")
print("    https://learn.microsoft.com/power-bi/create-reports/copilot-prepare-data-ai")
print("  Data Agents (optional):")
print("    https://learn.microsoft.com/fabric/data-science/data-agent-semantic-model")
print("  Semantic Link Labs:")
print("    https://github.com/microsoft/semantic-link-labs")
print("=" * 75)
print()
print("✅ Analysis complete! Review results and complete manual checks above.")
print("💾 Save this notebook for future reference and re-run after making changes.")

---
### 📝 Notes

**About This Analyzer**
This notebook validates semantic models for AI-powered analytics including:
- **Power BI Copilot** (in Power BI Service and Desktop)
- **Fabric Data Agents** (for multi-source AI solutions)

All technical checks apply to both Copilot and Data Agents — they both rely on the same Prep for AI configuration (AI Data Schema, Verified Answers, AI Instructions, synonyms, row labels).

**Version History**

**v1.0 (Based on Data Agent Readiness v2.2)**
- Restructured for general AI/Copilot usage (not Data Agent-specific)
- Check 1.17: Performance Analyzer for AI Data Schema measures (manual)
- Check 1.18: Duplicate column names across tables ✅ Automated
- Security requirements (item 6) in Section 0 scoping checklist
- Default groupings & analysis preferences (item 7) in Section 4 AI Instructions checklist
- Section 5 renamed to "Copilot & AI Testing" with testing guidance

**Manual Checks Required:**
- Memory Analyzer (run separately)
- Performance Analyzer for AI Data Schema measures (1.17)
- AI Data Schema configuration
- Verified Answers creation
- AI Instructions configuration (incl. default groupings)
- Security requirements definition
- Copilot/Data Agent testing
- Testing & validation workflow

**Automated Checks:**
- Star schema validation ✅
- Best Practice Analyzer ✅
- Data types ✅
- Naming conventions ✅
- Descriptions ✅
- Synonyms ✅
- Row labels ✅
- Implicit measures ✅
- Duplicates (measures) ✅
- Duplicate column names across tables ✅
- Helper measures ✅
- Measure dependencies ✅

---
*This notebook does not modify your semantic model. Run it periodically to validate readiness before deploying Copilot or Data Agent features.*